In [1]:
!pip install pyspark

In [2]:
!java -version

[0.028s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.028s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ETL_MercadoLaboral") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.4


In [6]:
from google.colab import files
uploaded = files.upload()

Saving salarios_experiencia.csv to salarios_experiencia.csv


In [7]:
salarios_spark = spark.read.csv("salarios_experiencia.csv", header=True, inferSchema=True)
salarios_spark.printSchema()
salarios_spark.show(5)

root
 |-- job_id: long (nullable = true)
 |-- formatted_experience_level: string (nullable = true)
 |-- normalized_salary: double (nullable = true)

+---------+--------------------------+-----------------+
|   job_id|formatted_experience_level|normalized_salary|
+---------+--------------------------+-----------------+
|   921716|                   Unknown|          38480.0|
| 10998357|                   Unknown|          55000.0|
| 23221523|                   Unknown|         157500.0|
| 91700727|                   Unknown|          35360.0|
|103254301|                   Unknown|         180000.0|
+---------+--------------------------+-----------------+
only showing top 5 rows


Cargado correctamente con Spark. Ahora calculamos la métrica: salario mediano por nivel de experiencia, usando groupBy de Spark.

In [8]:
from pyspark.sql import functions as F

resultado = salarios_spark.groupBy("formatted_experience_level") \
    .agg(
        F.expr("percentile_approx(normalized_salary, 0.5)").alias("salario_mediano"),
        F.count("*").alias("num_ofertas")
    ) \
    .orderBy("salario_mediano")

resultado.show(truncate=False)

+--------------------------+---------------+-----------+
|formatted_experience_level|salario_mediano|num_ofertas|
+--------------------------+---------------+-----------+
|Internship                |48880.0        |1346       |
|Entry level               |52000.0        |33499      |
|Associate                 |72500.0        |8594       |
|Unknown                   |77500.0        |25364      |
|Mid-Senior level          |105950.0       |36880      |
|Director                  |165000.0       |3274       |
|Executive                 |190000.0       |1038       |
+--------------------------+---------------+-----------+

